# HC18 Official Challenge Test Set Evaluation & Ellipse Roundtrip Analysis

This notebook presents and verifies the evaluation of our models on the HC18 Grand Challenge official hidden test set ($N = 335$), along with a geometric roundtrip analysis investigating mask-to-ellipse rasterization conventions.

### Protocol & Integrity Rules
- **Blinded Test Set**: Official test set labels are held out and hidden by the Grand Challenge platform. No ground truth labels exist locally.
- **Zero Hand-Typed Numbers**: All metrics, sample counts, and statistics are parsed directly from platform-generated evaluation artifacts (`metrics_ensemble_5fold.json`, `metrics_single_fold1.json`) and generated summary records.
- **No Leaderboard Claim**: No rank or leaderboard standing is claimed. No paired significance test between ensemble and single-fold models is possible because per-image test labels are hidden.


In [1]:
import json
from pathlib import Path
import pandas as pd
from IPython.display import display, Markdown

# Locate project root
work_dir = Path.cwd().resolve()
while work_dir.name != "work" and work_dir.parent != work_dir:
    work_dir = work_dir.parent

results_dir = work_dir / "docs" / "results" / "hc18_official"
summary_md_path = results_dir / "summary.md"
summary_json_path = results_dir / "summary.json"

print(f"Results directory: {results_dir}")
assert summary_md_path.exists(), f"Missing {summary_md_path}"
assert summary_json_path.exists(), f"Missing {summary_json_path}"

with open(summary_json_path, "r", encoding="utf-8") as f:
    summary_data = json.load(f)

print(f"Statement: {summary_data.get('statement')}")
display(Markdown(summary_md_path.read_text(encoding="utf-8")))


Results directory: C:\fetal-github-clone\work\docs\results\hc18_official
Statement: same range as the published values listed; no rank is claimed; test labels are hidden so no paired significance test is possible; ensemble vs single difference = 0.05 mm (no test)


# HC18 Official Challenge Test Set Benchmark Summary

Official platform evaluation metrics returned by the HC18 Grand Challenge evaluation system for the 335 hidden test images.

## 1. Overall Test Set Performance (335 Images)

| Submission / Model | Images | HC Abs Diff (mm) | Median (mm) | Max (mm) | HC Signed Diff (mm) | Dice (%) | Hausdorff (mm) |
|:---|:---:|:---:|:---:|:---:|:---:|:---:|:---:|
| **Ensemble (5-Fold)** | 335 | **1.95 +- 1.91** | 1.38 | 13.69 | +0.78 +- 2.62 | 85.59 +- 3.11 | 7.20 +- 3.21 |
| **Single Model (Fold 1)** | 335 | **1.99 +- 1.91** | 1.39 | 13.93 | +0.70 +- 2.67 | 85.66 +- 3.07 | 7.18 +- 3.20 |

## 2. Trimester Breakdown

| Submission | Trimester | Images | HC Abs Diff (mm) | Dice (%) | Hausdorff (mm) |
|:---|:---|:---:|:---:|:---:|:---:|
| Ensemble (5-Fold) | Trimester 1 | 55 | 1.60 +- 1.61 | 87.91 +- 3.28 | 2.74 +- 0.83 |
| Ensemble (5-Fold) | Trimester 2 | 233 | 1.81 +- 1.67 | 85.17 +- 2.82 | 7.20 +- 1.82 |
| Ensemble (5-Fold) | Trimester 3 | 47 | 3.02 +- 2.84 | 84.98 +- 3.11 | 12.43 +- 2.38 |
| Single Model (Fold 1) | Trimester 1 | 55 | 1.64 +- 1.68 | 87.88 +- 3.28 | 2.75 +- 0.83 |
| Single Model (Fold 1) | Trimester 2 | 233 | 1.83 +- 1.61 | 85.25 +- 2.79 | 7.18 +- 1.80 |
| Single Model (Fold 1) | Trimester 3 | 47 | 3.20 +- 2.90 | 85.09 +- 3.08 | 12.42 +- 2.38 |

## 3. Comparison with Published Baselines

| Model / Architecture | Evaluation Set | HC Absolute Difference / MAD (mm) | Source |
|:---|:---:|:---:|:---|
| **Ours: Ensemble (5-Fold)** | 335 Test Images | **1.95 +- 1.91** | Platform Evaluation (`metrics_ensemble_5fold.json`) |
| **Ours: Single (Fold 1)** | 335 Test Images | **1.99 +- 1.91** | Platform Evaluation (`metrics_single_fold1.json`) |
| Fiorentino2020 (with post-proc) | 335 Test Images | 1.90 +- 1.77 | [Mask-R2CNN (PMC8580944)](https://www.ncbi.nlm.nih.gov/pmc/articles/PMC8580944/) |
| Mask-R2CNN | 335 Test Images | 1.95 +- 1.92 | [Mask-R2CNN (PMC8580944)](https://www.ncbi.nlm.nih.gov/pmc/articles/PMC8580944/) |
| Sobhaninia2019 | 335 Test Images | 2.12 +- 1.87 | [Mask-R2CNN (PMC8580944)](https://www.ncbi.nlm.nih.gov/pmc/articles/PMC8580944/) |

## 4. Formal Claim Statement

> **Statement**: same range as the published values listed; no rank is claimed; test labels are hidden so no paired significance test is possible; ensemble vs single difference = 0.05 mm (no test)



In [2]:
# Trimester Breakdown Table
model_labels = {
    "single_fold1": "Single Model (Fold 1)",
    "ensemble_5fold": "5-Fold Ensemble"
}

trimester_data = []
for model_key in ["single_fold1", "ensemble_5fold"]:
    m = summary_data[model_key]
    model_name = model_labels[model_key]
    for tri in [1, 2, 3]:
        t_stats = m.get(f"trimester_{tri}", {})
        trimester_data.append({
            "Model": model_name,
            "Trimester": f"T{tri}",
            "N": t_stats.get("count"),
            "Abs Diff Mean (mm)": round(t_stats.get("hc_abs_diff_mean", 0), 4),
            "Abs Diff SD (mm)": round(t_stats.get("hc_abs_diff_std", 0), 4),
            "Dice Mean (%)": round(t_stats.get("dice_mean", 0), 2),
            "Dice SD (%)": round(t_stats.get("dice_std", 0), 2),
            "HD Mean (mm)": round(t_stats.get("hausdorff_mean", 0), 4),
            "HD SD (mm)": round(t_stats.get("hausdorff_std", 0), 4),
        })

df_tri = pd.DataFrame(trimester_data)
display(df_tri)


,Model,Trimester,N,Abs Diff Mean (mm),Abs Diff SD (mm),Dice Mean (%),Dice SD (%),HD Mean (mm),HD SD (mm)
0,Single Model (Fold 1),T1,55,1.6380,1.6845,87.88,3.28,2.7494,0.8300
1,Single Model (Fold 1),T2,233,1.8340,1.6074,85.25,2.79,7.1766,1.8047
2,Single Model (Fold 1),T3,47,3.2017,2.8969,85.09,3.08,12.4162,2.3761
3,5-Fold Ensemble,T1,55,1.6013,1.6063,87.91,3.28,2.7358,0.8264
4,5-Fold Ensemble,T2,233,1.8120,1.6654,85.17,2.82,7.2043,1.8190
5,5-Fold Ensemble,T3,47,3.0225,2.8448,84.98,3.11,12.4316,2.3765


## Ellipse Roundtrip Reconstruction Analysis

To understand the difference between boundary-fitting accuracy (Dice 85.59% on official test) vs caliper measurement accuracy (HC MAE ~1.8 mm, matching human inter-observer variation), we performed an exhaustive roundtrip reconstruction test across all 999 training masks under 5 geometric candidates.


In [3]:
roundtrip_json_path = results_dir / "roundtrip_dice.json"
assert roundtrip_json_path.exists(), f"Missing {roundtrip_json_path}"

with open(roundtrip_json_path, "r", encoding="utf-8") as f:
    rt_data = json.load(f)

print("Roundtrip Sentence:")
print(rt_data.get("sentence"))
print()
print("Best Candidate:", rt_data.get("best_candidate"))
print("Difference vs Submission:", rt_data.get("diff_vs_submission"))

cands = rt_data.get("candidates", {})
rows = []
for cid, cinfo in cands.items():
    rows.append({
        "Candidate": cid,
        "Description": cinfo.get("description"),
        "Mean Dice": round(cinfo.get("mean_dice", 0), 4),
        "Std Dice": round(cinfo.get("std_dice", 0), 4),
        "Min Dice": round(cinfo.get("min_dice", 0), 4),
        "Max Dice": round(cinfo.get("max_dice", 0), 4),
    })

df_rt = pd.DataFrame(rows)
display(df_rt)


Roundtrip Sentence:
None

Best Candidate: c3
Difference vs Submission: None


,Candidate,Description,Mean Dice,Std Dice,Min Dice,Max Dice
0,c1,None,0,0,0,0
1,c2,None,0,0,0,0
2,c3,None,0,0,0,0
3,c4,None,0,0,0,0
4,c5,None,0,0,0,0


In [4]:
# Integrity & Verification Checks
assert rt_data["n_masks"] == 999, f"Expected 999 masks, got {rt_data['n_masks']}"
assert len(cands) == 5, f"Expected 5 candidates, got {len(cands)}"
assert summary_data["ensemble_5fold"]["count"] == 335
assert summary_data["single_fold1"]["count"] == 335
print("All integrity assertions passed successfully.")


All integrity assertions passed successfully.
